In [38]:
import numpy as np
import pyuvdata

In [39]:
# Load an example file to get metadata
uv = pyuvdata.UVData()
uv.read("/Users/ruby/Downloads/uncalib_data_9828.uvfits")
uv.phase_to_time(np.mean(uv.time_array))  # Ensure the data are phased to zenith. This modifies the uvws.

# Zero out data
uv.data_array[...] = 0 + 0*1j

In [40]:
# Define sources to simulate
n_sources = 10

# Random source positions across the visibile hemisphere
source_azimuths = np.random.uniform(low=0.0, high=2*np.pi, size=n_sources)
source_zenith_angles = (1 - np.random.uniform(low=0.0, high=1, size=n_sources)**2.0) * np.pi/2

# Random source intensities. A uniform distribution is not the most physical.
source_intensities = np.random.uniform(low=0.0, high=10.0, size=n_sources)  

In [41]:
# Convert source positions into plane projected coordinates (l, m)
l = np.sin(source_zenith_angles) * np.cos(source_azimuths)
m = np.sin(source_zenith_angles) * np.sin(source_azimuths)

In [42]:
# Simulate sources. Use a simple DFT and ignore w terms.
wavelength = 3e8 / np.mean(uv.freq_array)
uv.data_array[:, 0, 0] = np.sum(
    source_intensities[:, np.newaxis] * np.exp(
        2
        * np.pi
        * 1j
        * (uv.uvw_array[np.newaxis, :, 0] * l[:, np.newaxis] + uv.uvw_array[np.newaxis, :, 1] * m[:, np.newaxis])
        / wavelength
    ), axis=0
)

In [43]:
uv.write_uvfits("/Users/ruby/Downloads/data_calibrated.uvfits")

antnums_to_baseline: found antenna numbers > 255, using 2048 baseline indexing.
Found antenna numbers > 255 in this data set. This is permitted by UVFITS standards, but may cause the `importuvfits` utility within CASA to crash. If attempting to use this data set in CASA, consider using the measurement set writer method (`write_ms`) instead.


In [44]:
# Generate random gains
cal = pyuvdata.UVCal()
cal.read("/Users/ruby/Downloads/gains_9999.calfits") # Load an example calibration table

gain_amp_stddev = 0.1
cal.gain_array = (
    np.random.normal(loc=1.0, scale=gain_amp_stddev, size=np.shape(cal.gain_array)) 
    * np.exp(1j*np.random.uniform(low=0, high=2*np.pi, size=np.shape(cal.gain_array)))
)
cal.gain_convention = "multiply"
new_uv = pyuvdata.utils.uvcalibrate(uv, cal, inplace=False, time_check=False)
new_uv.write_uvfits("/Users/ruby/Downloads/data_uncalibrated.uvfits")
cal.gain_convention = "divide"
cal.write_calfits("/Users/ruby/Downloads/gains.calfits")

gain_scale is not set, so there is no way to know what the resulting units are. For now, we assume that `gain_scale` matches whatever is on the UVData object (i.e. we do not change its units). Furthermore, all corrections concerning the pol_convention will be ignored.
pol_convention is not specified on the UVCal object, and uvc_pol_convention was not specified. Tentatively assuming that the UVCal and UVData objects (implicitly) have the same convention.
Neither uvd_pol_convention nor uvc_pol_convention are specified, so the resulting UVData object will have ambiguous convention. 
antnums_to_baseline: found antenna numbers > 255, using 2048 baseline indexing.
Found antenna numbers > 255 in this data set. This is permitted by UVFITS standards, but may cause the `importuvfits` utility within CASA to crash. If attempting to use this data set in CASA, consider using the measurement set writer method (`write_ms`) instead.
